In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import wbgapi as wb
from fredapi import Fred
from dotenv import load_dotenv
import os

load_dotenv()

print("All libraries loaded!")
print(f"pandas:   {pd.__version__}")
print(f"numpy:    {np.__version__}")
print(f"yfinance: {yf.__version__}")

All libraries loaded!
pandas:   3.0.0
numpy:    2.4.1
yfinance: 1.5.1


In [2]:
# WHY yfinance download for Nikkei:
# ^N225 is Yahoo Finance's ticker symbol for nikkie 225
# yfinance pulls historical daily prices directly - no manual

nikkei = yf.download('^N225', start='2014-01-01', end='2023-12-31')

print(f"Nikkei shape: {nikkei.shape}")
print(f"\nColumns: {nikkei.columns.tolist()}")
print(f"\nDate range: {nikkei.index.min()} to {nikkei.index.max()}")
nikkei.head()

[*********************100%***********************]  1 of 1 completed

Nikkei shape: (2444, 5)

Columns: [('Close', '^N225'), ('High', '^N225'), ('Low', '^N225'), ('Open', '^N225'), ('Volume', '^N225')]

Date range: 2014-01-06 00:00:00 to 2023-12-29 00:00:00


Price,Close,High,Low,Open,Volume
Ticker,^N225,^N225,^N225,^N225,^N225
Date,,,,,
2014-01-06,15908.879883,16164.009766,15864.440430,16147.540039,192700000
2014-01-07,15814.370117,15935.370117,15784.250000,15835.410156,165900000
2014-01-08,16121.450195,16121.450195,15906.570312,15943.679688,206700000
2014-01-09,15880.330078,16004.559570,15838.440430,16002.879883,217400000
2014-01-10,15912.059570,15922.139648,15754.700195,15785.150391,237500000


In [3]:
# WHY these two specific companies:
# Toyota = Japan's largest exporter, automotive sector proxy
# Sony = Japan's electronics/entertainment giant

toyota = yf.download('TM', start='2014-01-01', end='2023-12-31')
sony = yf.download('SONY', start='2014-01-01', end='2023-12-31')

print(f"Toyota shape: {toyota.shape}")
print(f"Sony shape: {sony.shape}")
print(f"\nToyota columns: {toyota.columns.tolist()}")

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

Toyota shape: (2516, 5)
Sony shape: (2516, 5)

Toyota columns: [('Close', 'TM'), ('High', 'TM'), ('Low', 'TM'), ('Open', 'TM'), ('Volume', 'TM')]


In [7]:
# WHY flattern MultiIndex columns:
# Right now columns look like ('Close', '^N225') - a tuple
# When the time of merging stock data with GDP data (single-level)
# columns like 'gdp_usd'), mixing MultiIndex and single-level 
# columns in the same merge causes confusing KeyErrors.
# Flattening now means every DataFrame speaks the same

def flatten_columns(df, ticker_name):
    df = df.copy()
    df.columns = [col[0] for col in df.columns]
    df['ticker'] = ticker_name
    df = df.reset_index()
    return df

nikkei_clean = flatten_columns(nikkei, 'Nikkei225')
toyota_clean = flatten_columns(toyota, 'Toyota')
sony_clean = flatten_columns(sony, 'sony')

print("Nikkei columns after flatten:", nikkei_clean.columns.tolist())
print("\nSample:")
nikkei_clean.head(3)

Nikkei columns after flatten: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'ticker']

Sample:


,Date,Close,High,Low,Open,Volume,ticker
0,2014-01-06,15908.879883,16164.009766,15864.440430,16147.540039,192700000,Nikkei225
1,2014-01-07,15814.370117,15935.370117,15784.250000,15835.410156,165900000,Nikkei225
2,2014-01-08,16121.450195,16121.450195,15906.570312,15943.679688,206700000,Nikkei225


In [9]:
# WHY save raw API pulls to data/raw/:
# Even though we're using live APIs (not manual downloads),
# saving a snapshot means: (1) we can work offline later, (2) reproducibility - someone can see exactly what data version analyzed in this project
# (3) if Yahoo Finance changes historical data or has downtime, we still have our original pull

nikkei_clean.to_csv("../data/raw/nikkei_raw.csv", index=False)
toyota_clean.to_csv("../data/raw/toyota_raw.csv", index=False)
sony_clean.to_csv("../data/raw/sony_raw.csv", index=False)